# Session 1 · What a Language Model Actually Does
*Understanding Language Models* · a course

**In this session you will:** watch a language model break text into pieces, see the odds it gives to every possible next word, and run a small experiment showing that these machines learn *genre and style* before they learn *truth*.

**Time:** about 60–75 minutes.

### How to use this notebook
- This is a **Google Colab notebook**: a page that mixes reading with small pieces of code you can run.
- To run a grey code box, click it and press **Shift + Enter** (or click the ▶ button on its left).
- **Run the boxes in order, top to bottom.** If something breaks, go to *Runtime → Restart session* and start again from the top.
- You never have to *write* code. Where you see text inside quotation marks, like `"this"`, you can change the words and run the box again. That is the whole skill.
- Boxes marked **Setup** load the machinery. You can open them if you are curious, but you do not need to read them.

## The big idea

A useful starting point is a small story from 2019. An early model called GPT-2 was given the opening of a fake news report about scientists discovering unicorns in the Andes, and it wrote a convincing science article: a named researcher, a university, quotes from the field, the careful tone of a science journalist. It also contradicted itself, describing the unicorns as having four horns.

The point is that the excitement had little to do with "intelligence". What impressed everyone was the **genre**: the machine reproduced the shape of a science article almost perfectly while failing at consistency and truth. The wider claim is that these systems capture **language as a cultural system** (genre, style, patterns, clichés) before anything like reasoning.

Today we test that claim ourselves, using GPT-2, the very model from that story.

## Setup

In [ ]:
#@title Setup: load GPT-2 (takes about a minute)
import torch, textwrap
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed

tok = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2")
model.eval()

def show_tokens(text):
    """Show how the model chops text into tokens."""
    ids = tok(text)["input_ids"]
    pieces = [tok.decode([i]) for i in ids]
    print(f"{len(pieces)} tokens:\n")
    print(" | ".join(repr(p)[1:-1] for p in pieces))

def next_word_table(text, k=10):
    """Show the model's top-k guesses for the next token, with probabilities."""
    ids = tok(text, return_tensors="pt")["input_ids"]
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    top = torch.topk(probs, k)
    rows = [(tok.decode([int(i)]), f"{float(p)*100:.1f}%")
            for p, i in zip(top.values, top.indices)]
    return pd.DataFrame(rows, columns=["next token", "probability"])

def word_by_word(text, steps=10):
    """Always pick the single most likely next token, and show each step."""
    for _ in range(steps):
        ids = tok(text, return_tensors="pt")["input_ids"]
        with torch.no_grad():
            nxt = int(model(ids).logits[0, -1].argmax())
        piece = tok.decode([nxt])
        text += piece
        print(f"+ {piece!r:<14} →  {text}")

def generate(prompt, length=120, temperature=0.9, seed=None):
    """Let the model continue a prompt, with some randomness."""
    if seed is not None:
        set_seed(seed)
    ids = tok(prompt, return_tensors="pt")
    out = model.generate(**ids, max_new_tokens=length, do_sample=True,
                         temperature=temperature, top_p=0.95,
                         pad_token_id=tok.eos_token_id)
    text = tok.decode(out[0], skip_special_tokens=True)
    print(textwrap.fill(text, 90))
    return text

print("GPT-2 is loaded and ready.")

## Part 1 · Language as pieces: tokens

A model does not see letters or words the way we do. It sees **tokens**: chunks of text from a fixed vocabulary of about 50,000 pieces. Common words are usually one token. Rarer words, names and many non-English words get chopped into several.

Run the box below. Then change the sentence and run it again.

In [ ]:
show_tokens("The curator hung the paintings in the new gallery.")

In [ ]:
# Try: a name, a place you know well, a word in another language, a made-up word.
show_tokens("Kente weavers in Bonwire use patterns called adweneasa.")

**Notice:** which words stay whole and which get broken up? The vocabulary was built from mostly English internet text, so the model literally has a finer grip on some languages and names than others. We come back to this in Session 5.

## Part 2 · The only thing it does: guess the next piece

Everything a language model does comes from one operation: given the text so far, assign a probability to every possible next token. Run the box to see the top ten guesses.

In [ ]:
next_word_table("The museum was closed because of the")

In [ ]:
# Change the opening and watch the guesses change.
next_word_table("After the concert, the drummer walked out into the")

**Try these openings** (change the text in quotes above):
- the first half of a proverb you know
- `"Once upon a"` versus `"Dear Sir or"` versus `"BREAKING:"`
- the opening words of a grant application: `"This project aims to"`

Each opening switches on a different **genre**, and the guesses change completely. The model is not looking anything up. It is recognising which kind of text it is inside.

## Part 3 · Building text one piece at a time

Chatbots write by repeating that single guess over and over. Below, the model always takes its top guess and adds it to the text. Watch the sentence grow.

In [ ]:
word_by_word("The exhibition opens next week and", steps=12)

Always picking the top guess tends to produce safe, repetitive, slightly dull text. Real systems add a little randomness, which is what we do next.

## Part 4 · The unicorn experiment

Here is an opening in the spirit of the famous 2019 example. It describes something impossible, so the model **cannot** be reporting facts. Anything convincing it produces must come from its grip on the genre.

Run it three times (or change the `seed` number) to get different versions.

In [ ]:
prompt = ("Scientists announced today that they had discovered a colony of penguins "
          "living deep in the Namib Desert. Even more surprising to the research team, "
          "the penguins were able to sing in four-part harmony.")

story = generate(prompt, length=150, seed=1)

In [ ]:
story = generate(prompt, length=150, seed=2)

### Worksheet: read it like a critic

For one of the stories, note down:

1. **Genre markers.** Which features tell you "this is a science news article"? (named experts, institutions, quotes, hedged claims, particular vocabulary...)
2. **Consistency.** Does it contradict itself? Change numbers, names or places halfway?
3. **Truth.** Is there anything a fact-checker could verify?
4. **Form before content.** Did the model get the *form* right more reliably than the *content*?

Now write your own impossible opening in a different genre (a gallery wall text, a football match report, an obituary, a recipe) and run it.

In [ ]:
my_prompt = "Write your own impossible opening here, in the genre of your choice."
story = generate(my_prompt, length=150)

## Part 5 · Temperature: how adventurous should it be?

**Temperature** controls how willing the model is to pick less likely tokens. Low is cautious and repetitive; high is surprising and eventually incoherent.

In [ ]:
opening = "The best thing about living in a city full of artists is"
for t in [0.3, 1.0, 1.6]:
    print(f"\n=== temperature {t} ===")
    generate(opening, length=60, temperature=t, seed=7)

**Question:** where on this dial does "creativity" live? Is the most interesting text also the most probable one? Keep this in mind for Session 5, where we look at what the model produces when it plays it safe.

## Discussion

1. Before today, what did you imagine was happening when a chatbot answers? What has changed?
2. This course argues that the theory of language most people carry around is wrong: we assume words mainly *point at things in the world*, with style and genre added on top. The machine seems to learn it the other way round. Does that match your experience as someone who works with language, images or sound?
3. If a machine can produce convincing genre with no access to truth, what does that mean for press releases, wall texts, reviews and funding bids in your sector?

## Glossary
- **Token**: a chunk of text the model treats as one unit.
- **Next-token prediction**: the single task these models are trained on.
- **Probability**: how likely the model thinks each next token is.
- **Temperature**: a dial for how much randomness is allowed.
- **GPT-2**: an early (2019) model by OpenAI, small enough to run here for free. Modern chatbots are far larger but work on the same principle.

## Going further
- OpenAI's 2019 post "Better Language Models and Their Implications", where the unicorn story first appeared.